In [16]:
from pathlib import Path
import pandas as pd
meteo_dir = Path("气象数据/30分钟")
years = range(2003, 2011)
meteo_frames = []
for year in years:
    file_path = meteo_dir / f"{year}年禹城气象30分钟数据.xlsx"
    df_year = pd.read_excel(file_path)
    df_year = df_year.iloc[1:].reset_index(drop=True)
    df_year["来源年份"] = year
    meteo_frames.append(df_year)
df_meteo_30min = pd.concat(meteo_frames, ignore_index=True)
df_meteo_30min.head()

,年,月,日,时,分,秒,近地面空气温度,冠层上方空气温度,近地面空气湿度,冠层上方空气湿度,...,一层土壤温度,二层土壤温度,三层土壤温度,四层土壤温度,五层土壤温度,一层土壤体积含水量,二层土壤体积含水量,三层土壤体积含水量,降水量,来源年份
0,2003,1,1,0,30,0,-15.34,-15.27,89.5,89.6,...,-2.808,-0.38,1.429,3.245,6.972,0.351,0.427,-99999,0,2003
1,2003,1,1,1,0,0,-15.56,-15.5,89.1,89.2,...,-2.833,-0.385,1.427,3.243,6.97,0.351,0.427,-99999,0,2003
2,2003,1,1,1,30,0,-15.98,-15.88,88.7,88.8,...,-2.908,-0.391,1.424,3.239,6.966,0.35,0.427,-99999,0,2003
3,2003,1,1,2,0,0,-16.36,-16.27,88.3,88.4,...,-3.004,-0.397,1.42,3.235,6.963,0.35,0.427,-99999,0,2003
4,2003,1,1,2,30,0,-16.49,-16.36,88.4,88.6,...,-3.062,-0.403,1.416,3.232,6.961,0.349,0.427,-99999,0,2003


In [17]:
flux_dir = Path("通量数据/30分钟")
flux_frames = []
for year in years:
    file_path = flux_dir / f"{year}年禹城通量30分钟数据.xlsx"
    df_year = pd.read_excel(file_path)
    df_year = df_year.iloc[1:].reset_index(drop=True)
    df_year["来源年份"] = year
    flux_frames.append(df_year)
df_flux_30min = pd.concat(flux_frames, ignore_index=True)
df_flux_30min.head()

,年,月,日,时,分,NEE,RE,GEE,LE,Hs,来源年份
0,2003,1,1,0,0,0.020622,0.020622,0,12.476468,-12.534316,2003
1,2003,1,1,0,30,0.020405,0.020405,0,12.476468,-12.534316,2003
2,2003,1,1,1,0,0.020323,0.020323,0,12.476468,-12.534316,2003
3,2003,1,1,1,30,0.02008,0.02008,0,12.476468,-12.534316,2003
4,2003,1,1,2,0,0.019771,0.019771,0,12.476468,-12.534316,2003


In [18]:
# 3. 高效按日期对齐气象与通量数据
import numpy as np
date_columns = ["年", "月", "日", "时", "分"]
def prepare_half_hourly(df):
    numeric = df[date_columns].apply(pd.to_numeric, errors="coerce")
    dated = pd.to_datetime({
        "year": numeric["年"],
        "month": numeric["月"],
        "day": numeric["日"],
        "hour": numeric["时"],
        "minute": numeric["分"],
    }, errors="coerce")
    return (
        df.assign(date_time=dated)
        .drop(columns=date_columns + ["秒"], errors="ignore")
        .dropna(subset=["date_time"])
    )
df_meteo_prepared = prepare_half_hourly(df_meteo_30min)
df_flux_prepared = prepare_half_hourly(df_flux_30min)
df_merged = pd.merge(
    df_meteo_prepared,
    df_flux_prepared,
    on=["date_time", "来源年份"],
    how="inner",
    suffixes=("_meteo", "_flux"),
)
df_merged.sort_values(["date_time", "来源年份"], inplace=True)
df_merged.head()

,近地面空气温度,冠层上方空气温度,近地面空气湿度,冠层上方空气湿度,近地面水汽压,冠层上方水汽压,近地面风速,冠层上方风速,风向,大气压,...,二层土壤体积含水量,三层土壤体积含水量,降水量,来源年份,date_time,NEE,RE,GEE,LE,Hs
0,-15.34,-15.27,89.5,89.6,0.145,0.146,0,0.01,214.5,102.9,...,0.427,-99999,0,2003,2003-01-01 00:30:00,0.020405,0.020405,0,12.476468,-12.534316
1,-15.56,-15.5,89.1,89.2,0.142,0.143,0.019,0.06,25.33,102.9,...,0.427,-99999,0,2003,2003-01-01 01:00:00,0.020323,0.020323,0,12.476468,-12.534316
2,-15.98,-15.88,88.7,88.8,0.136,0.138,0,0.005,359.1,102.9,...,0.427,-99999,0,2003,2003-01-01 01:30:00,0.02008,0.02008,0,12.476468,-12.534316
3,-16.36,-16.27,88.3,88.4,0.131,0.132,0,0,0,103,...,0.427,-99999,0,2003,2003-01-01 02:00:00,0.019771,0.019771,0,12.476468,-12.534316
4,-16.49,-16.36,88.4,88.6,0.13,0.131,0,0.001,136.6,103,...,0.427,-99999,0,2003,2003-01-01 02:30:00,0.019587,0.019587,0,12.476468,-12.534316


In [19]:
# 4. 筛选保留的字段并清除无关指标
columns_to_exclude = {"来源年份", "RE", "GEE", "LE", "Hs", "ET","H","三层土壤体积含水量"}
suffixes = ("_meteo", "_flux")
drop_candidates = set()
for col in columns_to_exclude:
    drop_candidates.add(col)
    for suffix in suffixes:
        drop_candidates.add(f"{col}{suffix}")
# Excel 导入常见的未命名列也一并移除
drop_candidates.update(
    column for column in df_merged.columns if column.lower().startswith("unnamed")
)
columns_removed = [column for column in df_merged.columns if column in drop_candidates]
df_selected = df_merged.drop(columns=columns_removed, errors="ignore")
print("Removed columns:", columns_removed[:10],"..." if len(columns_removed) > 10 else "")
df_selected.head()

Removed columns: ['三层土壤体积含水量', '来源年份', 'RE', 'GEE', 'LE', 'Hs'] 


,近地面空气温度,冠层上方空气温度,近地面空气湿度,冠层上方空气湿度,近地面水汽压,冠层上方水汽压,近地面风速,冠层上方风速,风向,大气压,...,一层土壤温度,二层土壤温度,三层土壤温度,四层土壤温度,五层土壤温度,一层土壤体积含水量,二层土壤体积含水量,降水量,date_time,NEE
0,-15.34,-15.27,89.5,89.6,0.145,0.146,0,0.01,214.5,102.9,...,-2.808,-0.38,1.429,3.245,6.972,0.351,0.427,0,2003-01-01 00:30:00,0.020405
1,-15.56,-15.5,89.1,89.2,0.142,0.143,0.019,0.06,25.33,102.9,...,-2.833,-0.385,1.427,3.243,6.97,0.351,0.427,0,2003-01-01 01:00:00,0.020323
2,-15.98,-15.88,88.7,88.8,0.136,0.138,0,0.005,359.1,102.9,...,-2.908,-0.391,1.424,3.239,6.966,0.35,0.427,0,2003-01-01 01:30:00,0.02008
3,-16.36,-16.27,88.3,88.4,0.131,0.132,0,0,0,103,...,-3.004,-0.397,1.42,3.235,6.963,0.35,0.427,0,2003-01-01 02:00:00,0.019771
4,-16.49,-16.36,88.4,88.6,0.13,0.131,0,0.001,136.6,103,...,-3.062,-0.403,1.416,3.232,6.961,0.349,0.427,0,2003-01-01 02:30:00,0.019587


In [20]:
temp_cols = ["一层土壤温度", "二层土壤温度", "三层土壤温度"]
extra_temp_cols = ["四层土壤温度", "五层土壤温度"]
moisture_cols = ["一层土壤体积含水量", "二层土壤体积含水量"]
for column_group, new_name in [
    (temp_cols, "土壤温度"),
    (moisture_cols, "土壤含水量"),
]:
    existing = [column for column in column_group if column in df_selected.columns]
    if not existing:
        print(f"Warning: 未找到 {new_name} 所需的列，跳过。")
        continue
    averaged = (
        df_selected[existing]
        .apply(pd.to_numeric, errors="coerce")
        .mean(axis=1)
    )
    df_selected[new_name] = averaged
    df_selected = df_selected.drop(columns=existing, errors="ignore")
df_selected = df_selected.drop(columns=extra_temp_cols, errors="ignore")
print("已生成列:", [name for _, name in [(temp_cols, "土壤温度"), (moisture_cols, "土壤含水量")]])
df_selected.head()

已生成列: ['土壤温度', '土壤含水量']


,近地面空气温度,冠层上方空气温度,近地面空气湿度,冠层上方空气湿度,近地面水汽压,冠层上方水汽压,近地面风速,冠层上方风速,风向,大气压,太阳辐射,净辐射,光合有效辐射,降水量,date_time,NEE,土壤温度,土壤含水量
0,-15.34,-15.27,89.5,89.6,0.145,0.146,0,0.01,214.5,102.9,0,-9.19,0,0,2003-01-01 00:30:00,0.020405,-0.586333,0.3890
1,-15.56,-15.5,89.1,89.2,0.142,0.143,0.019,0.06,25.33,102.9,0,-10.93,0,0,2003-01-01 01:00:00,0.020323,-0.597000,0.3890
2,-15.98,-15.88,88.7,88.8,0.136,0.138,0,0.005,359.1,102.9,0,-11.61,0,0,2003-01-01 01:30:00,0.02008,-0.625000,0.3885
3,-16.36,-16.27,88.3,88.4,0.131,0.132,0,0,0,103,0,-11.24,0,0,2003-01-01 02:00:00,0.019771,-0.660333,0.3885
4,-16.49,-16.36,88.4,88.6,0.13,0.131,0,0.001,136.6,103,0,-10.46,0,0,2003-01-01 02:30:00,0.019587,-0.683000,0.3880


In [21]:
# 6. 调整列顺序：保留完整时间戳（含时分）
if "date_time" not in df_selected.columns:
    raise KeyError("df_selected 中缺少 date_time 列，无法生成 date")
df_selected["date"] = pd.to_datetime(df_selected["date_time"], errors="coerce")
df_selected = df_selected.drop(columns=["date_time"], errors="ignore")
priority_columns = ["date"]
trailing_columns = ["NEE"]
existing_priority = [column for column in priority_columns if column in df_selected.columns]
existing_trailing = [column for column in trailing_columns if column in df_selected.columns]
middle_columns = [
    column
    for column in df_selected.columns
    if column not in existing_priority + existing_trailing
]
new_order = existing_priority + middle_columns + existing_trailing
df_selected = df_selected.reindex(columns=new_order)
df_selected.head()

,date,近地面空气温度,冠层上方空气温度,近地面空气湿度,冠层上方空气湿度,近地面水汽压,冠层上方水汽压,近地面风速,冠层上方风速,风向,大气压,太阳辐射,净辐射,光合有效辐射,降水量,土壤温度,土壤含水量,NEE
0,2003-01-01 00:30:00,-15.34,-15.27,89.5,89.6,0.145,0.146,0,0.01,214.5,102.9,0,-9.19,0,0,-0.586333,0.3890,0.020405
1,2003-01-01 01:00:00,-15.56,-15.5,89.1,89.2,0.142,0.143,0.019,0.06,25.33,102.9,0,-10.93,0,0,-0.597000,0.3890,0.020323
2,2003-01-01 01:30:00,-15.98,-15.88,88.7,88.8,0.136,0.138,0,0.005,359.1,102.9,0,-11.61,0,0,-0.625000,0.3885,0.02008
3,2003-01-01 02:00:00,-16.36,-16.27,88.3,88.4,0.131,0.132,0,0,0,103,0,-11.24,0,0,-0.660333,0.3885,0.019771
4,2003-01-01 02:30:00,-16.49,-16.36,88.4,88.6,0.13,0.131,0,0.001,136.6,103,0,-10.46,0,0,-0.683000,0.3880,0.019587


In [22]:
# 8. 基于互信息与树模型寻找非线性关系（按指定列顺序）
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
candidate_columns = [
    "近地面空气温度",
    "冠层上方空气温度",
    "近地面空气湿度",
    "冠层上方空气湿度",
    "近地面水汽压",
    "冠层上方水汽压",
    "近地面风速",
    "冠层上方风速",
    "风向",
    "大气压",
    "太阳辐射",
    "净辐射",
    "光合有效辐射",
    "H",
    "土壤温度",
    "土壤含水量",
]
target_column = "NEE"
if target_column not in df_selected.columns:
    raise ValueError("df_selected 中缺少 NEE 列，无法执行非线性分析。")
available_features = [
    column
    for column in candidate_columns
    if column in df_selected.columns
]
if not available_features:
    raise ValueError("候选特征均未在 df_selected 中找到，请检查列名。")
X_numeric = df_selected[available_features].apply(pd.to_numeric, errors="coerce")
valid_features = [column for column in X_numeric.columns if not X_numeric[column].isna().all()]
if not valid_features:
    raise ValueError("候选特征均为空，无法计算非线性关系。")
analysis_frame = pd.concat([
    df_selected[[target_column]].apply(pd.to_numeric, errors="coerce"),
    X_numeric[valid_features],
], axis=1).dropna()
if analysis_frame.empty:
    raise ValueError("非线性分析需要的列为空，请检查缺失值或列名。")
X = analysis_frame[valid_features]
y = analysis_frame[target_column]
mi_scores = mutual_info_regression(X, y, random_state=42)
mi_series = (
    pd.Series(mi_scores, index=valid_features)
    .sort_values(ascending=False)
)
print("互信息得分 (完整列表):")
print(mi_series.to_string())
top_mi_features = mi_series.head(6).index.tolist()
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)
rf.fit(X_train, y_train)
rf_importances = pd.Series(
    rf.feature_importances_, index=valid_features
).sort_values(ascending=False)
print("随机森林特征重要度 (完整列表):")
print(rf_importances.to_string())

互信息得分 (完整列表):
土壤温度        0.778380
土壤含水量       0.500200
近地面空气温度     0.479443
冠层上方空气温度    0.476297
净辐射         0.456629
大气压         0.425333
太阳辐射        0.418346
光合有效辐射      0.418166
近地面水汽压      0.395301
冠层上方水汽压     0.389125
近地面空气湿度     0.171972
冠层上方空气湿度    0.164772
近地面风速       0.156527
风向          0.153833
冠层上方风速      0.145696
随机森林特征重要度 (完整列表):
净辐射         0.469370
土壤含水量       0.067469
土壤温度        0.062077
近地面空气湿度     0.061153
近地面水汽压      0.060066
大气压         0.051981
光合有效辐射      0.038449
太阳辐射        0.037515
近地面空气温度     0.034781
近地面风速       0.025393
风向          0.024157
冠层上方空气温度    0.020798
冠层上方水汽压     0.018475
冠层上方风速      0.016717
冠层上方空气湿度    0.011598


In [23]:
# 9. 输出协变量两两相关并按 |corr| 从高到低排序，并导出为Excel
from pathlib import Path
feature_candidates = [
    column
    for column in df_selected.columns
    if column not in {"date", "NEE"}
 ]
X_numeric = df_selected[feature_candidates].apply(pd.to_numeric, errors="coerce")
feature_columns = [
    column
    for column in X_numeric.columns
    if not X_numeric[column].isna().all()
 ]
if not feature_columns:
    raise ValueError("未找到可用于计算相关性的协变量。")
X = X_numeric[feature_columns]
corr_matrix = X.corr()
pairs = []
for row in corr_matrix.index:
    for col in corr_matrix.columns:
        if row >= col:
            continue
        pairs.append((row, col, corr_matrix.loc[row, col]))
correlation_df = pd.DataFrame(
    pairs,
    columns=["feature_1", "feature_2", "pearson_corr"],
)
correlation_df["abs_corr"] = correlation_df["pearson_corr"].abs()
correlation_df = correlation_df.sort_values(
    by="abs_corr", ascending=False
)
display(correlation_df.drop(columns=["abs_corr"]))

# 导出为 Excel（去除辅助列 abs_corr），保存到当前目录
export_df = correlation_df.drop(columns=["abs_corr"])
out_path = Path("YuCheng_correlations.xlsx")
export_df.to_excel(out_path, index=False)


,feature_1,feature_2,pearson_corr
97,光合有效辐射,太阳辐射,1.000000
56,冠层上方风速,近地面风速,1.000000
4,冠层上方空气温度,近地面空气温度,0.999536
41,冠层上方水汽压,近地面水汽压,0.999261
57,冠层上方风速,风向,0.999145
...,...,...,...
95,光合有效辐射,风向,0.000656
59,冠层上方风速,太阳辐射,0.000190
94,光合有效辐射,冠层上方风速,0.000190
74,太阳辐射,近地面风速,0.000185


In [24]:
# 统计各列缺失/异常值（NaN、-9999、字符串NA等）并导出
from pathlib import Path
import pandas as pd
import numpy as np

if 'df_selected' not in globals():
    raise RuntimeError("需要已生成的 df_selected（含各列数据）。请先运行上方清洗流程。")

df = df_selected.copy()
total_rows = len(df)
na_tokens = {"na", "n/a", "nan", "null", "none", ""}
summary_rows = []

for col in df.columns:
    s = df[col]
    # NaN（pandas 缺失）
    mask_nan = s.isna()
    nan_count = int(mask_nan.sum())
    # -9999（数值或可转为数值的字符串）
    s_num = pd.to_numeric(s, errors="coerce")
    mask_minus9999 = s_num.eq(-99999)
    minus9999_count = int(mask_minus9999.sum())
    # 字符串形式的 NA（大小写及前后空白不敏感）
    s_str = s.astype(str).str.strip().str.lower()
    mask_na_str = s_str.isin(na_tokens)
    na_str_count = int(mask_na_str.sum())
    # 汇总“问题值”（三者并集）
    mask_problem = mask_nan | mask_minus9999 | mask_na_str
    problem_count = int(mask_problem.sum())
    # 统计行
    summary_rows.append({
        "column": col,
        "total_rows": total_rows,
        "nan_count": nan_count,
        "nan_rate": round(nan_count / total_rows, 6) if total_rows else 0.0,
        "minus9999_count": minus9999_count,
        "minus9999_rate": round(minus9999_count / total_rows, 6) if total_rows else 0.0,
        "na_string_count": na_str_count,
        "na_string_rate": round(na_str_count / total_rows, 6) if total_rows else 0.0,
        "problem_count": problem_count,
        "problem_rate": round(problem_count / total_rows, 6) if total_rows else 0.0,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(by=["problem_rate", "nan_rate", "minus9999_rate", "na_string_rate"], ascending=False).reset_index(drop=True)
display(summary_df)

# 导出为 Excel 和 CSV
excel_path = Path("missing_summary.xlsx")
csv_path = Path("missing_summary.csv")
summary_df.to_excel(excel_path, index=False)
summary_df.to_csv(csv_path, index=False)
print("已导出缺失统计:", excel_path, "和", csv_path)

,column,total_rows,nan_count,nan_rate,minus9999_count,minus9999_rate,na_string_count,na_string_rate,problem_count,problem_rate
0,风向,139322,0,0.0,12278,0.088127,0,0.0,12278,0.088127
1,近地面风速,139322,0,0.0,12259,0.087990,0,0.0,12259,0.087990
2,冠层上方风速,139322,0,0.0,12259,0.087990,0,0.0,12259,0.087990
3,土壤含水量,139322,0,0.0,10718,0.076930,0,0.0,10718,0.076930
4,降水量,139322,0,0.0,1193,0.008563,0,0.0,1193,0.008563
5,土壤温度,139322,0,0.0,864,0.006201,0,0.0,864,0.006201
6,date,139322,0,0.0,0,0.000000,0,0.0,0,0.000000
7,近地面空气温度,139322,0,0.0,0,0.000000,0,0.0,0,0.000000
8,冠层上方空气温度,139322,0,0.0,0,0.000000,0,0.0,0,0.000000
9,近地面空气湿度,139322,0,0.0,0,0.000000,0,0.0,0,0.000000


已导出缺失统计: missing_summary.xlsx 和 missing_summary.csv


In [25]:
# 9. 删除指定的协变量列
columns_to_remove = [
    "太阳辐射",
    "光合有效辐射",
    "冠层上方空气湿度",
    "近地面风速",
    "降水量",
    "冠层上方空气温度",
    "冠层上方水汽压",
]
existing_to_remove = [column for column in columns_to_remove if column in df_selected.columns]
if existing_to_remove:
    df_selected = df_selected.drop(columns=existing_to_remove)
    print("已删除列:", existing_to_remove)
else:
    print("未找到需要删除的列。")
df_selected.head()

已删除列: ['太阳辐射', '光合有效辐射', '冠层上方空气湿度', '近地面风速', '降水量', '冠层上方空气温度', '冠层上方水汽压']


,date,近地面空气温度,近地面空气湿度,近地面水汽压,冠层上方风速,风向,大气压,净辐射,土壤温度,土壤含水量,NEE
0,2003-01-01 00:30:00,-15.34,89.5,0.145,0.01,214.5,102.9,-9.19,-0.586333,0.3890,0.020405
1,2003-01-01 01:00:00,-15.56,89.1,0.142,0.06,25.33,102.9,-10.93,-0.597000,0.3890,0.020323
2,2003-01-01 01:30:00,-15.98,88.7,0.136,0.005,359.1,102.9,-11.61,-0.625000,0.3885,0.02008
3,2003-01-01 02:00:00,-16.36,88.3,0.131,0,0,103,-11.24,-0.660333,0.3885,0.019771
4,2003-01-01 02:30:00,-16.49,88.4,0.13,0.001,136.6,103,-10.46,-0.683000,0.3880,0.019587


In [26]:
# 在所有数值列中去除包含-99999的行
value_cols = [c for c in df_selected.columns if c != "date"]
mask_minus99999 = df_selected[value_cols].apply(lambda col: pd.to_numeric(col, errors="coerce").eq(-99999), axis=0).any(axis=1)
rows_before = len(df_selected)
df_selected = df_selected[~mask_minus99999].copy()
rows_removed = rows_before - len(df_selected)
print(f"剔除含 -99999 的行数: {rows_removed}")
df_selected.head()

剔除含 -99999 的行数: 16518


,date,近地面空气温度,近地面空气湿度,近地面水汽压,冠层上方风速,风向,大气压,净辐射,土壤温度,土壤含水量,NEE
0,2003-01-01 00:30:00,-15.34,89.5,0.145,0.01,214.5,102.9,-9.19,-0.586333,0.3890,0.020405
1,2003-01-01 01:00:00,-15.56,89.1,0.142,0.06,25.33,102.9,-10.93,-0.597000,0.3890,0.020323
2,2003-01-01 01:30:00,-15.98,88.7,0.136,0.005,359.1,102.9,-11.61,-0.625000,0.3885,0.02008
3,2003-01-01 02:00:00,-16.36,88.3,0.131,0,0,103,-11.24,-0.660333,0.3885,0.019771
4,2003-01-01 02:30:00,-16.49,88.4,0.13,0.001,136.6,103,-10.46,-0.683000,0.3880,0.019587


In [27]:
# 统计各列缺失/异常值（NaN、-9999、字符串NA等）并导出
from pathlib import Path
import pandas as pd
import numpy as np

if 'df_selected' not in globals():
    raise RuntimeError("需要已生成的 df_selected（含各列数据）。请先运行上方清洗流程。")

df = df_selected.copy()
total_rows = len(df)
na_tokens = {"na", "n/a", "nan", "null", "none", ""}
summary_rows = []

for col in df.columns:
    s = df[col]
    # NaN（pandas 缺失）
    mask_nan = s.isna()
    nan_count = int(mask_nan.sum())
    # -9999（数值或可转为数值的字符串）
    s_num = pd.to_numeric(s, errors="coerce")
    mask_minus9999 = s_num.eq(-99999)
    minus9999_count = int(mask_minus9999.sum())
    # 字符串形式的 NA（大小写及前后空白不敏感）
    s_str = s.astype(str).str.strip().str.lower()
    mask_na_str = s_str.isin(na_tokens)
    na_str_count = int(mask_na_str.sum())
    # 汇总“问题值”（三者并集）
    mask_problem = mask_nan | mask_minus9999 | mask_na_str
    problem_count = int(mask_problem.sum())
    # 统计行
    summary_rows.append({
        "column": col,
        "total_rows": total_rows,
        "nan_count": nan_count,
        "nan_rate": round(nan_count / total_rows, 6) if total_rows else 0.0,
        "minus9999_count": minus9999_count,
        "minus9999_rate": round(minus9999_count / total_rows, 6) if total_rows else 0.0,
        "na_string_count": na_str_count,
        "na_string_rate": round(na_str_count / total_rows, 6) if total_rows else 0.0,
        "problem_count": problem_count,
        "problem_rate": round(problem_count / total_rows, 6) if total_rows else 0.0,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(by=["problem_rate", "nan_rate", "minus9999_rate", "na_string_rate"], ascending=False).reset_index(drop=True)
display(summary_df)

# 导出为 Excel 和 CSV
excel_path = Path("missing_summary.xlsx")
csv_path = Path("missing_summary.csv")
summary_df.to_excel(excel_path, index=False)
summary_df.to_csv(csv_path, index=False)
print("已导出缺失统计:", excel_path, "和", csv_path)

,column,total_rows,nan_count,nan_rate,minus9999_count,minus9999_rate,na_string_count,na_string_rate,problem_count,problem_rate
0,date,122804,0,0.0,0,0.0,0,0.0,0,0.0
1,近地面空气温度,122804,0,0.0,0,0.0,0,0.0,0,0.0
2,近地面空气湿度,122804,0,0.0,0,0.0,0,0.0,0,0.0
3,近地面水汽压,122804,0,0.0,0,0.0,0,0.0,0,0.0
4,冠层上方风速,122804,0,0.0,0,0.0,0,0.0,0,0.0
5,风向,122804,0,0.0,0,0.0,0,0.0,0,0.0
6,大气压,122804,0,0.0,0,0.0,0,0.0,0,0.0
7,净辐射,122804,0,0.0,0,0.0,0,0.0,0,0.0
8,土壤温度,122804,0,0.0,0,0.0,0,0.0,0,0.0
9,土壤含水量,122804,0,0.0,0,0.0,0,0.0,0,0.0


已导出缺失统计: missing_summary.xlsx 和 missing_summary.csv


In [28]:
# 10. 仅执行列重命名（不做数值转换、插值或缺失填充）
col_map = {
    "date": "date",
    "近地面空气温度": "Ta",
    "近地面空气湿度": "RH",
    "近地面水汽压": "VPD",
    "冠层上方风速": "WS",
    "风向": "WD",
    "大气压": "PA",
    "净辐射": "Rn",
    "土壤温度": "Ts",
    "土壤含水量": "SWC",
    "NEE": "NEE",
}
existing_map = {old: new for old, new in col_map.items() if old in df_selected.columns}
df_selected = df_selected.rename(columns=existing_map)
print("已重命名列:", existing_map)
df_selected.head()

已重命名列: {'date': 'date', '近地面空气温度': 'Ta', '近地面空气湿度': 'RH', '近地面水汽压': 'VPD', '冠层上方风速': 'WS', '风向': 'WD', '大气压': 'PA', '净辐射': 'Rn', '土壤温度': 'Ts', '土壤含水量': 'SWC', 'NEE': 'NEE'}


,date,Ta,RH,VPD,WS,WD,PA,Rn,Ts,SWC,NEE
0,2003-01-01 00:30:00,-15.34,89.5,0.145,0.01,214.5,102.9,-9.19,-0.586333,0.3890,0.020405
1,2003-01-01 01:00:00,-15.56,89.1,0.142,0.06,25.33,102.9,-10.93,-0.597000,0.3890,0.020323
2,2003-01-01 01:30:00,-15.98,88.7,0.136,0.005,359.1,102.9,-11.61,-0.625000,0.3885,0.02008
3,2003-01-01 02:00:00,-16.36,88.3,0.131,0,0,103,-11.24,-0.660333,0.3885,0.019771
4,2003-01-01 02:30:00,-16.49,88.4,0.13,0.001,136.6,103,-10.46,-0.683000,0.3880,0.019587


In [29]:
# 对除 NEE 外的数值列保留4位小数，NEE保留8位；剔除含科学计数法(e/E)的行
from pandas.api.types import is_numeric_dtype
import pandas as pd
import numpy as np

if 'df_selected' not in globals():
    raise RuntimeError("需要已生成的 df_selected。")

# 1) 剔除含 e/E 的行（排除日期列）
value_cols = [c for c in df_selected.columns if c != "date"]
mask_e_any = df_selected[value_cols].apply(
    lambda col: col.astype(str).str.contains(r"[eE]", regex=True, na=False)
).any(axis=1)
rows_before = len(df_selected)
df_selected = df_selected[~mask_e_any].copy()
rows_removed = rows_before - len(df_selected)
print(f"剔除含 e/E 科学计数法的行数: {rows_removed}")

# 2) 统一数值化后再四舍五入：非NEE保留4位，NEE保留8位
for col in value_cols:
    # 将可转换的内容转为数值（无法转换的设为 NaN）
    df_selected[col] = pd.to_numeric(df_selected[col], errors="coerce")
    # 仅对数值列执行四舍五入
    if is_numeric_dtype(df_selected[col]):
        if col == "NEE":
            df_selected[col] = df_selected[col].round(8)
        else:
            df_selected[col] = df_selected[col].round(4)
print("完成小数位处理：非NEE→4位，NEE→8位。")
df_selected.head()

剔除含 e/E 科学计数法的行数: 27
完成小数位处理：非NEE→4位，NEE→8位。


,date,Ta,RH,VPD,WS,WD,PA,Rn,Ts,SWC,NEE
0,2003-01-01 00:30:00,-15.34,89.5,0.145,0.010,214.50,102.9,-9.19,-0.5863,0.3890,0.020405
1,2003-01-01 01:00:00,-15.56,89.1,0.142,0.060,25.33,102.9,-10.93,-0.5970,0.3890,0.020323
2,2003-01-01 01:30:00,-15.98,88.7,0.136,0.005,359.10,102.9,-11.61,-0.6250,0.3885,0.020080
3,2003-01-01 02:00:00,-16.36,88.3,0.131,0.000,0.00,103.0,-11.24,-0.6603,0.3885,0.019771
4,2003-01-01 02:30:00,-16.49,88.4,0.130,0.001,136.60,103.0,-10.46,-0.6830,0.3880,0.019587


In [30]:
# 11. 导出处理后的数据集
from pathlib import Path
output_path = Path("YuCheng.csv")
df_selected.to_csv(output_path, index=False)
print("已导出:", output_path)

已导出: YuCheng.csv
